# Evaluation Findings: Phase 55 Feature & Signal Evaluation

This notebook synthesizes all evaluation results from Phase 55:

1. **IC Feature Rankings** — Top/bottom features by |IC-IR| across all assets/TFs
2. **Regime Analysis** — Regime-conditional IC heatmaps (BTC 1D)
3. **Experiment Results** — BH gate summary and top experimental features
4. **Adaptive RSI Comparison** — Static vs adaptive RSI A/B results
5. **Lifecycle Decisions** — Feature promotion recommendations

**Source data:** `cmc_ic_results` (82,110 rows), `cmc_feature_experiments` (67,788 rows)
**Date:** 2026-02-26
**Phase:** 55-05 (Evaluation Findings)

---

## Table of Contents

1. [Setup](#setup)
2. [Parameters](#parameters)
3. [IC Feature Rankings](#ic-rankings)
4. [IC Decay Chart](#ic-decay)
5. [Regime Analysis](#regime)
6. [Experiment Results](#experiments)
7. [Adaptive RSI Comparison](#adaptive-rsi)
8. [Lifecycle Summary](#lifecycle)
9. [Next Steps](#next-steps)


<a id='setup'></a>
## 1. Setup

In [ ]:
import sys
from pathlib import Path

_ROOT = Path.cwd().parent
if str(_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(_ROOT / "src"))

import helpers
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ta_lab2
print(f"ta_lab2 loaded from: {ta_lab2.__file__}")
print(f"Python: {sys.version}")
print(f"Pandas: {pd.__version__}")


In [ ]:
engine = helpers.get_engine()
print("DB connection OK")


<a id='parameters'></a>
## 2. Parameters

In [ ]:
# Parameters — adjust to explore different assets/timeframes
ASSET_ID     = 1        # BTC
TF           = "1D"
HORIZON      = 1        # 1-bar horizon for regime analysis
RETURN_TYPE  = "arith"  # arithmetic returns

REPORTS_DIR  = _ROOT / "reports" / "evaluation"

print(f"Asset  : {ASSET_ID} (BTC)")
print(f"TF     : {TF}")
print(f"Horizon: {HORIZON}")


### Load CSV Artifacts

In [ ]:
ic_ranking       = pd.read_csv(REPORTS_DIR / "ic_ranking_full.csv")
bh_gate          = pd.read_csv(REPORTS_DIR / "bh_gate_results.csv")
experiment_df    = pd.read_csv(REPORTS_DIR / "experiment_results.csv")
promo_decisions  = pd.read_csv(REPORTS_DIR / "promotion_decisions.csv")
adaptive_ic_csv  = pd.read_csv(REPORTS_DIR / "adaptive_rsi_ic_comparison.csv")
adaptive_wf_csv  = pd.read_csv(REPORTS_DIR / "adaptive_rsi_bakeoff.csv")

print(f"ic_ranking      : {len(ic_ranking)} features")
print(f"bh_gate         : {len(bh_gate)} features")
print(f"experiment_df   : {len(experiment_df)} rows")
print(f"promo_decisions : {len(promo_decisions)} features")
print(f"adaptive_ic_csv : {len(adaptive_ic_csv)} rows")
print(f"adaptive_wf_csv : {len(adaptive_wf_csv)} rows")


<a id='ic-rankings'></a>
## 3. IC Feature Rankings

### Top 20 Features by |IC-IR|

Features ranked by mean absolute IC-IR across all assets and TFs at horizon=1.
Higher |IC-IR| = more consistent predictive signal relative to noise.


In [ ]:
top20 = ic_ranking.head(20).copy()
top20["rank"] = range(1, len(top20) + 1)

fig = px.bar(
    top20,
    x="mean_abs_ic_ir",
    y="feature",
    orientation="h",
    color="mean_abs_ic_ir",
    color_continuous_scale="Blues",
    title="Top 20 Features by Mean |IC-IR| (All Assets, All TFs, H=1)",
    labels={
        "mean_abs_ic_ir": "Mean |IC-IR|",
        "feature": "Feature",
    },
    hover_data=["mean_abs_ic", "n_observations", "n_asset_tf_pairs"],
)
fig.update_layout(
    height=600,
    yaxis={"categoryorder": "total ascending"},
    showlegend=False,
    coloraxis_showscale=False,
)
fig.show()


### Bottom 20 Features (Deprecation Candidates)

In [ ]:
bottom20 = ic_ranking.tail(20).copy()
bottom20["rank"] = range(len(ic_ranking) - len(bottom20) + 1, len(ic_ranking) + 1)

fig = px.bar(
    bottom20,
    x="mean_abs_ic_ir",
    y="feature",
    orientation="h",
    color="mean_abs_ic_ir",
    color_continuous_scale="Reds_r",
    title="Bottom 20 Features by Mean |IC-IR| (Deprecation Candidates)",
    labels={
        "mean_abs_ic_ir": "Mean |IC-IR|",
        "feature": "Feature",
    },
)
fig.update_layout(
    height=600,
    yaxis={"categoryorder": "total ascending"},
    showlegend=False,
    coloraxis_showscale=False,
)
fig.show()


<a id='ic-decay'></a>
## 4. IC Decay Chart

IC Decay shows how the predictive signal deteriorates over longer horizons.
Strong features maintain |IC| at longer horizons (slow decay).
Sharp decay features are only useful for short-term prediction.


In [ ]:
from sqlalchemy import text as _text

# Load IC across horizons for top-10 features (BTC 1D, arith, all regime)
top10_features = ic_ranking.head(10)["feature"].tolist()

with engine.connect() as conn:
    decay_df = pd.read_sql(
        _text("""
            SELECT feature, horizon,
                   AVG(ABS(ic)) as mean_abs_ic,
                   AVG(ABS(ic_ir)) as mean_abs_ic_ir,
                   COUNT(*) as n_obs
            FROM cmc_ic_results
            WHERE asset_id = 1 AND tf = '1D'
              AND return_type = 'arith'
              AND regime_col = 'all' AND regime_label = 'all'
            GROUP BY feature, horizon
            ORDER BY feature, horizon
        """),
        conn,
    )

# Filter to top features
decay_top = decay_df[decay_df["feature"].isin(top10_features)]

fig = go.Figure()
for feat in top10_features:
    df_f = decay_top[decay_top["feature"] == feat].sort_values("horizon")
    if len(df_f) == 0:
        continue
    fig.add_trace(go.Scatter(
        x=df_f["horizon"],
        y=df_f["mean_abs_ic"],
        mode="lines+markers",
        name=feat,
        hovertemplate=f"<b>{feat}</b><br>Horizon: %{{x}}<br>|IC|: %{{y:.4f}}<extra></extra>",
    ))

fig.update_layout(
    title="IC Decay by Horizon — Top 10 Features (BTC 1D, Arith)",
    xaxis_title="Horizon (bars)",
    yaxis_title="Mean |IC|",
    height=500,
    hovermode="x unified",
    legend=dict(x=1.01, y=0.99),
)
fig.show()


<a id='regime'></a>
## 5. Regime Analysis

### Regime-Conditional IC Heatmap (BTC 1D)

IC-IR shown per feature x regime label.
Green = strong positive signal, Red = strong contrarian signal, White = no signal.


In [ ]:
# Load regime-conditional IC for top 20 features
top20_names = ic_ranking.head(20)["feature"].tolist()

with engine.connect() as conn:
    regime_df = pd.read_sql(
        _text("""
            SELECT feature, regime_col, regime_label,
                   AVG(ic_ir) as mean_ic_ir,
                   AVG(ABS(ic_ir)) as mean_abs_ic_ir
            FROM cmc_ic_results
            WHERE asset_id = 1 AND tf = '1D' AND horizon = 1
              AND return_type = 'arith'
              AND regime_col != 'all'
            GROUP BY feature, regime_col, regime_label
        """),
        conn,
    )

regime_df["label"] = regime_df["regime_col"] + " / " + regime_df["regime_label"]

# Create pivot for heatmap
pivot_regime = regime_df[regime_df["feature"].isin(top20_names)].pivot_table(
    index="feature", columns="label", values="mean_ic_ir"
)
pivot_regime = pivot_regime.reindex(
    [f for f in top20_names if f in pivot_regime.index]
)

# Regime column order
col_order = [
    "trend_state / Up", "trend_state / Sideways", "trend_state / Down",
    "vol_state / High", "vol_state / Normal", "vol_state / Low",
]
col_order = [c for c in col_order if c in pivot_regime.columns]
pivot_regime = pivot_regime[col_order]

fig = go.Figure(data=go.Heatmap(
    z=pivot_regime.values,
    x=pivot_regime.columns.tolist(),
    y=pivot_regime.index.tolist(),
    colorscale="RdYlGn",
    zmid=0,
    zmin=-2,
    zmax=2,
    text=[[f"{v:.3f}" if not np.isnan(v) else "N/A"
           for v in row]
          for row in pivot_regime.values],
    texttemplate="%{text}",
    textfont={"size": 9},
    hoverongaps=False,
))

fig.update_layout(
    title="Regime-Conditional IC-IR Heatmap — BTC 1D H=1 (Top 20 Features)",
    xaxis_title="Regime",
    yaxis_title="Feature",
    height=650,
    yaxis={"tickfont": {"size": 10}},
)
fig.show()


<a id='experiments'></a>
## 6. Experiment Results

### BH Gate Pass/Fail Distribution

Features evaluated via `ExperimentRunner` (100 features x 5 TFs x 17 assets x 7 horizons x 2 return types).
BH gate: at least one BH-adjusted p-value < 0.05 across all hypotheses for the feature.


In [ ]:
# BH gate pie chart
bh_valid = bh_gate[bh_gate["n_experiments"] > 0].copy()
pass_count = bh_valid["bh_passes"].sum()
fail_count = len(bh_valid) - pass_count

fig = go.Figure(data=[go.Pie(
    labels=["BH Passes (promote/keep)", "BH Fails (deprecate candidate)"],
    values=[pass_count, fail_count],
    marker_colors=["#2ecc71", "#e74c3c"],
    textinfo="label+percent+value",
)])
fig.update_layout(
    title=f"BH Gate Results — {len(bh_valid)} Features Evaluated",
    height=400,
)
fig.show()


### IC vs n_significant Scatter (All Experimental Features)

In [ ]:
# Scatter: mean_abs_ic vs n_significant (size = n_experiments)
bh_plot = bh_valid.dropna(subset=["mean_abs_ic"]).copy()
bh_plot["bh_color"] = bh_plot["bh_passes"].map({True: "#2ecc71", False: "#e74c3c"})
bh_plot["bh_label"] = bh_plot["bh_passes"].map({True: "Pass", False: "Fail"})

fig = px.scatter(
    bh_plot,
    x="mean_abs_ic",
    y="n_significant",
    size="n_experiments",
    color="bh_label",
    color_discrete_map={"Pass": "#2ecc71", "Fail": "#e74c3c"},
    hover_name="feature_name",
    title="Experimental Features: Mean |IC| vs n_Significant (BH-corrected)",
    labels={
        "mean_abs_ic": "Mean |IC| (across all assets/TFs/horizons)",
        "n_significant": "n Significant (BH p < 0.05)",
        "n_experiments": "n Experiments",
        "bh_label": "BH Gate",
    },
    size_max=20,
)
fig.add_vline(x=0.02, line_dash="dash", line_color="gray",
              annotation_text="IC=0.02 threshold")
fig.update_layout(height=550)
fig.show()


### Top 10 Experimental Features by Mean |IC|

In [ ]:
top10_exp = bh_valid.sort_values("mean_abs_ic", ascending=False).head(10)

fig = px.bar(
    top10_exp,
    x="mean_abs_ic",
    y="feature_name",
    orientation="h",
    color="bh_passes",
    color_discrete_map={True: "#2ecc71", False: "#e74c3c"},
    title="Top 10 Experimental Features by Mean |IC|",
    labels={
        "mean_abs_ic": "Mean |IC|",
        "feature_name": "Feature",
        "bh_passes": "BH Passes",
    },
    hover_data=["n_experiments", "n_significant"],
)
fig.update_layout(
    height=450,
    yaxis={"categoryorder": "total ascending"},
)
fig.show()


<a id='adaptive-rsi'></a>
## 7. Adaptive RSI A/B Comparison

Comparison of static RSI (fixed 30/70 thresholds) vs adaptive RSI (rolling 100-bar 20th/80th percentile).
Source: Phase 55-04. Full analysis: `reports/evaluation/adaptive_rsi_ab_comparison.md`.


In [ ]:
# IC comparison: side-by-side bars per horizon
# adaptive_ic_csv has columns: asset, horizon, return_type, static_ic, static_ic_ir, adaptive_ic, adaptive_ic_ir
ic_comp = adaptive_ic_csv.copy()

# Filter to arith returns for clarity
ic_arith = ic_comp[ic_comp.get("return_type", ic_comp.columns[2]) == "arith"] if "return_type" in ic_comp.columns else ic_comp

# Use all rows if no return_type filter
if len(ic_arith) == 0:
    ic_arith = ic_comp

print(f"Adaptive IC comparison rows: {len(ic_arith)}")
print(ic_arith.columns.tolist())
ic_arith.head(10)


In [ ]:
# Plot |IC-IR| comparison
if "static_ic_ir" in ic_arith.columns and "adaptive_ic_ir" in ic_arith.columns:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["BTC: Static vs Adaptive |IC-IR| by Horizon",
                        "ETH: Static vs Adaptive |IC-IR| by Horizon"],
        shared_yaxes=True,
    )

    for col_idx, asset_label in enumerate(["BTC", "ETH"], 1):
        if "asset" in ic_arith.columns:
            df_asset = ic_arith[ic_arith["asset"] == asset_label]
        elif "asset_id" in ic_arith.columns:
            asset_id_val = 1 if asset_label == "BTC" else 1027
            df_asset = ic_arith[ic_arith["asset_id"] == asset_id_val]
        else:
            df_asset = ic_arith if col_idx == 1 else pd.DataFrame()

        if df_asset.empty:
            continue

        fig.add_trace(go.Bar(
            x=df_asset["horizon"],
            y=df_asset["static_ic_ir"].abs(),
            name="Static",
            marker_color="#3498db",
            showlegend=(col_idx == 1),
        ), row=1, col=col_idx)
        fig.add_trace(go.Bar(
            x=df_asset["horizon"],
            y=df_asset["adaptive_ic_ir"].abs(),
            name="Adaptive",
            marker_color="#e67e22",
            showlegend=(col_idx == 1),
        ), row=1, col=col_idx)

    fig.update_layout(
        title="Static vs Adaptive RSI |IC-IR| by Horizon (1D)",
        barmode="group",
        height=450,
        xaxis_title="Horizon (bars)",
        yaxis_title="|IC-IR|",
    )
    fig.show()
else:
    print("Column names:", ic_arith.columns.tolist())
    print("Need static_ic_ir and adaptive_ic_ir columns. Showing raw data:")
    display(ic_arith)


In [ ]:
# Walk-forward Sharpe comparison
wf = adaptive_wf_csv.copy()
print(f"Walk-forward bakeoff rows: {len(wf)}")
print(wf.columns.tolist())
wf.head()


In [ ]:
# Plot walk-forward Sharpe by fold
if "static_sharpe" in wf.columns and "adaptive_sharpe" in wf.columns:
    fold_col = "fold" if "fold" in wf.columns else wf.columns[0]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=wf[fold_col].astype(str),
        y=wf["static_sharpe"],
        name="Static RSI",
        marker_color="#3498db",
    ))
    fig.add_trace(go.Bar(
        x=wf[fold_col].astype(str),
        y=wf["adaptive_sharpe"],
        name="Adaptive RSI",
        marker_color="#e67e22",
    ))
    fig.add_hline(y=0, line_dash="solid", line_color="black", line_width=1)
    fig.update_layout(
        title="Walk-Forward OOS Sharpe: Static vs Adaptive RSI (BTC 1D)",
        xaxis_title="OOS Fold",
        yaxis_title="OOS Sharpe",
        barmode="group",
        height=450,
    )
    fig.show()

    # Print summary
    print(f"\nStatic RSI — Mean Sharpe: {wf['static_sharpe'].mean():.3f}, Mean MaxDD: {wf.get('static_maxdd', pd.Series([float('nan')])).mean():.3f}")
    print(f"Adaptive RSI — Mean Sharpe: {wf['adaptive_sharpe'].mean():.3f}")
    print(f"Decision: INCONCLUSIVE — Static retained (IC-IR: static wins 14/14)")
else:
    print("Column names:", wf.columns.tolist())
    display(wf)


<a id='lifecycle'></a>
## 8. Lifecycle Decisions Summary

Feature lifecycle recommendations from `promotion_decisions.csv`.
Based on: BH gate pass/fail + mean |IC-IR| thresholds (>0.03 = promote, 0.005-0.03 = keep, <0.005 or BH fail = deprecate).


In [ ]:
# Lifecycle distribution donut chart
lifecycle_counts = promo_decisions["recommendation"].value_counts()

fig = go.Figure(data=[go.Pie(
    labels=lifecycle_counts.index.tolist(),
    values=lifecycle_counts.values.tolist(),
    hole=0.4,
    marker_colors=["#2ecc71", "#f39c12", "#e74c3c"],
    textinfo="label+percent+value",
)])
fig.update_layout(
    title=f"Feature Lifecycle Recommendations ({len(promo_decisions)} features total)",
    height=400,
)
fig.show()


In [ ]:
# Promotion candidates: IC-IR scatter by recommendation
promo_plot = promo_decisions.dropna(subset=["mean_abs_ic_ir"]).copy()

color_map = {
    "promote": "#2ecc71",
    "keep_experimental": "#f39c12",
    "deprecate_candidate": "#e74c3c",
}

fig = px.scatter(
    promo_plot,
    x="ic_rank",
    y="mean_abs_ic_ir",
    color="recommendation",
    color_discrete_map=color_map,
    hover_name="feature_name",
    hover_data=["n_significant_bh", "bh_passes", "mean_abs_ic"],
    title="Feature Lifecycle Decisions by IC Rank",
    labels={
        "ic_rank": "IC Rank (1 = best)",
        "mean_abs_ic_ir": "Mean |IC-IR|",
        "recommendation": "Recommendation",
    },
)
fig.add_hline(y=0.03, line_dash="dash", line_color="#2ecc71",
              annotation_text="Promote threshold (IC-IR=0.03)")
fig.add_hline(y=0.005, line_dash="dash", line_color="#e74c3c",
              annotation_text="Deprecate threshold (IC-IR=0.005)")
fig.update_layout(height=550)
fig.show()


In [ ]:
# Summary table: promoted features
promoted = promo_decisions[promo_decisions["recommendation"] == "promote"].copy()
print(f"Features recommended for PROMOTION: {len(promoted)}")
print()
display(promoted[["feature_name", "ic_rank", "mean_abs_ic_ir", "n_significant_bh", "notes"]].head(25))


In [ ]:
# Summary table: deprecation candidates
deprecate = promo_decisions[promo_decisions["recommendation"] == "deprecate_candidate"].copy()
print(f"Features flagged as DEPRECATION CANDIDATES: {len(deprecate)}")
print()
display(deprecate[["feature_name", "ic_rank", "mean_abs_ic_ir", "n_significant_bh", "bh_passes"]].head(30))


<a id='next-steps'></a>
## 9. Next Steps

### Immediate Actions

1. **Run FeaturePromoter** for the 60 recommended-for-promotion features:
   ```python
   from ta_lab2.experiments import FeaturePromoter, FeatureRegistry
   registry = FeatureRegistry("configs/experiments/features.yaml")
   registry.load()
   promoter = FeaturePromoter(engine, registry)
   for feature in promoted_features:
       promoter.promote_feature(feature)
   ```

2. **Update features.yaml** — Set `lifecycle: deprecated` for deprecation candidates.

3. **Regime-conditional follow-up** — Re-run `borderline` features with regime conditioning enabled.

4. **Signal pipeline integration** — Promoted features → `cmc_features` → signal generators.

### Phase 56+ Roadmap

- **Phase 56**: Regime-stratified signal generation using promoted features
- **Phase 57**: Portfolio construction with regime-filtered signals
- **Phase 58**: Paper trading integration

---

_Notebook generated by Phase 55-05 execution plan. See `reports/evaluation/EVALUATION_FINDINGS.md` for full report._
